# ML-04 — Search Intelligence Data Contract

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-04 — Search Intelligence Data Contract


## 1. Unit of analysis + time window

**One row = one content page** (one client × one content item). The bundle is a fixed 90-day snapshot (period ending 2026-06-30 in the full release; the bundled CSV is the matching anonymized slice). I verify the grain and confirm no future window enters the label below.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows:", len(df), "duplicates:", df.duplicated(["content_id","client_id"]).sum())
print("columns:", len(df.columns))

# all 90-day and last/prev-30-day columns are backward looking by construction
backward = [c for c in df.columns if ("90d" in c) or ("prev_30d" in c) or ("last_30d" in c)]
print("backward-looking time columns present:", len(backward))

## 2. Fields: feature / label / context / excluded

- **Features:** `impressions_90d`, `impressions_last_30d`, `impressions_prev_30d`, `clicks_90d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `age_tier`, `days_since_last_update`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `ai_sessions_90d`, `impression_tier`, `position_tier`, `trend_pct`.
- **Label:** `is_declining_label` derived from `trend_direction`.
- **Context (not features):** `content_id`, `client_id` (grouping/ID only).
- **Excluded (and why):** `provider_used`, `model_used` — these describe *how the content was generated*, a product/business-internal choice that must not leak into triage; they carry no observable outcome signal for this lane. `trend_direction` itself is excluded from features (it defines the label).

## 3. Verify it with queries (grain, counts, missing values, windows)

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# grain
assert df.duplicated(["content_id","client_id"]).sum() == 0
print("grain OK: 1 row per content page")

# label balance
lbl = df["trend_direction"].eq("down").astype(int)
print("label rows:", lbl.sum(), f"({lbl.mean():.3f})")

# missing values per feature group
miss = df.isna().sum()
print("columns with >0 missing:", sorted(miss[miss>0].index.tolist()))
print("missing share of those:", {c: round(df[c].isna().mean(),3) for c in miss[miss>0].index[:6]})

## 4. Data limits

- This is a 90-day snapshot: **no cross-season trend**, no monthly seasonality.
- `impressions_prev_30d`/`last_30d` overlap with the 90-day window — I keep them as features but never read them as independent windows.
- The label ('trend_direction') is a shipped rule, not hand-labelled ground truth.
- The data can never tell me *why* a page declined (no page text, no query mix, no competitor actions), only that it did, and whether demand/engagement signals align.

## Self-check

- [x] Grain + window stated and verified
- [x] Field buckets complete with an exclusion reason
- [x] Queries back every claim
- [x] Limits written honestly